# Fine-tuning Qwen2.5-1.5B (QLoRA) for adverse drug event extraction
Detect adverse drug events (ADEs) in sentences from medical case reports and extract every (drug, effect) pair as JSON.

**Result on 1,000 held-out sentences:** ADE F1 0.54 → **0.90**, pair F1 0.26 → **0.70** (strict exact match).
Adapter: [AadarshKr/qwen2.5-1.5b-ade-lora](https://huggingface.co/AadarshKr/qwen2.5-1.5b-ade-lora).
See the README for the write-up.

*The training, adapter-saving and generation cells were not re-run in this saved version (to save GPU time); their results are recorded next to them and saved in the Hugging Face repo.*

## 0. Setup
Use a single GPU (the trainer does not support DataParallel). Run the install cell once per session, then **Run → Restart & clear cell outputs** before running the rest.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # make only one GPU visible

In [ ]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [2]:
!nvidia-smi

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

Sun Nov  8 15:07:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import transformers, trl, peft
print(transformers.__version__, trl.__version__, peft.__version__)

5.22.2 1.17.1 0.21.2


## 1. Load and explore the data
Dataset: `ade-benchmark-corpus/ade_corpus_v2` (classification + drug–ADE relation configs).

In [4]:
from datasets import load_dataset
import pandas as pd

ade_clf = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_classification")
ade_rel = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_drug_ade_relation")
clf_df = ade_clf["train"].to_pandas()
rel_df = ade_rel["train"].to_pandas()

README.md:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

Ade_corpus_v2_classification/train-00000(…): reconstructing file:   0%|          |  0.00B / 1.71MB            

Ade_corpus_v2_classification/train-00000(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/23516 [00:00<?, ? examples/s]

Ade_corpus_v2_drug_ade_relation/train-00(…): reconstructing file:   0%|          |  0.00B /  491kB            

Ade_corpus_v2_drug_ade_relation/train-00(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6821 [00:00<?, ? examples/s]

In [6]:
print(clf_df.shape)
print(clf_df.columns)
print(clf_df.head(5))

(23516, 2)
Index(['text', 'label'], dtype='object')
                                                text  label
0      Intravenous azithromycin-induced ototoxicity.      1
1  Immobilization, while Paget's bone disease was...      1
2  Unaccountable severe hypercalcemia in a patien...      1
3  METHODS: We report two cases of pseudoporphyri...      1
4  METHODS: We report two cases of pseudoporphyri...      1


In [7]:
print(rel_df.shape)
print(rel_df.columns)
print(rel_df.head(5))

(6821, 4)
Index(['text', 'drug', 'effect', 'indexes'], dtype='object')
                                                text                drug  \
0      Intravenous azithromycin-induced ototoxicity.        azithromycin   
1  Immobilization, while Paget's bone disease was...  dihydrotachysterol   
2  Unaccountable severe hypercalcemia in a patien...  dihydrotachysterol   
3  METHODS: We report two cases of pseudoporphyri...            naproxen   
4  METHODS: We report two cases of pseudoporphyri...           oxaprozin   

                      effect  \
0                ototoxicity   
1  increased calcium-release   
2              hypercalcemia   
3            pseudoporphyria   
4            pseudoporphyria   

                                             indexes  
0  {'drug': {'start_char': [12], 'end_char': [24]...  
1  {'drug': {'start_char': [91], 'end_char': [109...  
2  {'drug': {'start_char': [84], 'end_char': [102...  
3  {'drug': {'start_char': [58], 'end_char': [66]...  
4  {

In [8]:
print(clf_df["label"].value_counts())

label
0    16695
1     6821
Name: count, dtype: int64


In [9]:
print(rel_df.count())
print(rel_df["text"].nunique())
print(rel_df["text"].value_counts().max())

text       6821
drug       6821
effect     6821
indexes    6821
dtype: int64
4271
24


In [10]:
print(clf_df["text"].nunique())
print(clf_df.shape)

20896
(23516, 2)


In [11]:
uniq_lab1 = clf_df[clf_df["label"] == 1].nunique()
uniq_lab0 = clf_df[clf_df["label"] == 0].nunique()
print(uniq_lab1,'\n',uniq_lab0)

text     4271
label       1
dtype: int64 
 text     16625
label        1
dtype: int64


In [12]:
both_labels = (clf_df.groupby("text")["label"].nunique()>1).sum()
print(both_labels)

0


In [13]:
clf_ade = clf_df[clf_df["label"] == 1]
set_clf_ade = set(clf_ade["text"])
set_rel = set(rel_df["text"])
print(len(set_clf_ade & set_rel))
print(len(set_rel - set_clf_ade))

4271
0


**Findings**
- ADE sentences repeat once per drug–effect pair: 6,821 relation rows but only 4,271 unique sentences (max 24 rows for one sentence).
- 20,896 unique sentences in total; no sentence has conflicting labels; the ADE sentences in both configs are identical.
- After de-duplication the true ADE share is **20%** (4,271 / 20,896), not 29% as the raw rows suggest.

## 2. One clean record per sentence
Merge the two configs into one row per unique sentence: `text`, `label`, and the list of (drug, effect) `pairs`.

**De-duplicate the classification set**

In [14]:
uniq_clf_df = clf_df.drop_duplicates(subset = "text")
print(uniq_clf_df.shape)

(20896, 2)


**Group the drug–effect pairs by sentence**

In [15]:
grouped_rel_df = (
    rel_df
    .drop_duplicates(subset=["text", "drug", "effect"])
    .groupby("text")
    .apply(lambda x: x[["drug","effect"]].to_dict(orient = "records"),include_groups=False)
    .reset_index(name="pairs")
)
print(grouped_rel_df.shape)
grouped_rel_df.head(3)

(4271, 2)


,text,pairs
0,1. Changes in the plasma cortisol level were r...,"[{'drug': 'alprazolam', 'effect': 'Changes in ..."
1,"2-CdA induces lymphocytopenia, which may expla...","[{'drug': '2-CdA', 'effect': 'lymphocytopenia'}]"
2,2-CdA typically causes a long-lasting state of...,"[{'drug': '2-CdA', 'effect': 'immunodeficiency..."


**Sentence with the most pairs**

In [16]:
print(grouped_rel_df["pairs"].str.len().idxmax())
print(grouped_rel_df["pairs"].str.len().max())
print(grouped_rel_df.loc[31, "text"])

31
24
A 27-year-old man who had a history of bronchial asthma, eosinophilic enteritis, and eosinophilic pneumonia presented with fever, skin eruptions, cervical lymphadenopathy, hepatosplenomegaly, atypical lymphocytosis, and eosinophilia two weeks after receiving trimethoprim (TMP)-sulfamethoxazole (SMX) treatment.


Annotators paired every effect with every drug mention: 6 effects × 4 drug strings (trimethoprim, TMP, sulfamethoxazole, SMX) = 24 pairs.

**Merge** (non-ADE sentences get an empty list)

In [17]:
records = uniq_clf_df.merge(grouped_rel_df, on = "text", how = "left")
records["pairs"] = records["pairs"].apply(lambda v: v if isinstance(v,list) else [])
records.head(5)

,text,label,pairs
0,Intravenous azithromycin-induced ototoxicity.,1,"[{'drug': 'azithromycin', 'effect': 'ototoxici..."
1,"Immobilization, while Paget's bone disease was...",1,"[{'drug': 'dihydrotachysterol', 'effect': 'inc..."
2,Unaccountable severe hypercalcemia in a patien...,1,"[{'drug': 'dihydrotachysterol', 'effect': 'hyp..."
3,METHODS: We report two cases of pseudoporphyri...,1,"[{'drug': 'naproxen', 'effect': 'pseudoporphyr..."
4,"Naproxen, the most common offender, has been a...",1,"[{'drug': 'Naproxen', 'effect': 'erythropoieti..."


**Validation checks**

In [18]:
print(records.shape)

label1_rec = records[records["label"] == 1]
print(label1_rec.shape)

n_pairs = label1_rec["pairs"].str.len()
print("max pairs:",n_pairs.max())
print("mean pairs:",round(n_pairs.mean(),2))
print(((records["pairs"].str.len() == 0) & (records["label"] == 1)).sum())
print(((records["pairs"].str.len() != 0) & (records["label"] == 0)).sum())

(20896, 3)
(4271, 3)
max pairs: 24
mean pairs: 1.55
0
0


## 3. Leakage-safe, stratified split
Rows are unique sentences, so a row split is a sentence split and no sentence can appear in two sets. Stratified by label to keep the 20.4% ADE share in every set. The test set is used only for the final evaluation.

In [20]:
from sklearn.model_selection import train_test_split

temp_df, test_df = train_test_split(records, test_size = 1000, stratify = records["label"], random_state = 42)
print(temp_df.shape)

train_df, val_df = train_test_split(temp_df, test_size = 500, stratify = temp_df["label"], random_state = 42)
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

print("train ADE %:", round(train_df["label"].mean() * 100, 1))
print("val ADE %:", round(val_df["label"].mean() * 100, 1))
print("test ADE %:", round(test_df["label"].mean() * 100, 1))

print("train & test overlap:", len(set(train_df["text"]) & set(test_df["text"])))
print("train & val overlap:", len(set(train_df["text"]) & set(val_df["text"])))
print("val & test overlap:", len(set(val_df["text"]) & set(test_df["text"])))

(19896, 3)
train: 19396
val: 500
test: 1000
train ADE %: 20.4
val ADE %: 20.4
test ADE %: 20.4
train & test overlap: 0
train & val overlap: 0
val & test overlap: 0


## 4. Prompt and JSON target
Chat format: fixed system rules + the sentence (prompt), and the correct JSON answer (completion). The same prompt is used for the base and fine-tuned models; training loss is computed on the completion only.

In [21]:
import json

SYSTEM_PROMPT = (
    "You are a pharmacovigilance assistant. Read the sentence and decide whether it "
    "describes an adverse drug event (ADE): a harmful effect caused by a drug. "
    "If it does, list every (drug, effect) pair. "
    "Copy the drug and the effect exactly as they are written in the sentence. "
    "If there is no ADE, return an empty list of pairs. "
    'Return only JSON in this exact format: {"has_ade": true or false, "pairs": [{"drug": "...", "effect": "..."}]}'
)

def make_target(row):
    answer = {"has_ade": bool(row["label"]), "pairs": row["pairs"]}
    return json.dumps(answer, ensure_ascii=False)

def make_prompt(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Sentence: " + text}
    ]

def add_columns(df):
    df = df.copy()
    df["prompt"] = df["text"].apply(make_prompt)
    df["completion"] = df.apply(lambda row: [{"role":"assistant", "content": make_target(row)}], axis = 1)
    return df

train_df = add_columns(train_df)
val_df = add_columns(val_df)
test_df = add_columns(test_df)

ade_ex = train_df[train_df["label"] == 1].iloc[0]
non_ex = train_df[train_df["label"] == 0].iloc[0]
print(ade_ex["prompt"][1]["content"], "\n→", ade_ex["completion"][0]["content"], "\n")
print(non_ex["prompt"][1]["content"], "\n→", non_ex["completion"][0]["content"])

Sentence: We describe a patient with CREST syndrome (calcinosis, Raynaud's phenomenon, esophageal dysfunction, sclerodactyly, and telangiectasia) who paradoxically experienced worsening of Raynaud's phenomenon when using yohimbine for ED. 
→ {"has_ade": true, "pairs": [{"drug": "yohimbine", "effect": "worsening of Raynaud's phenomenon"}]} 

Sentence: A Meckel diverticulum is an embryonic remnant of the vitellointestinal duct. 
→ {"has_ade": false, "pairs": []}


## 5. Load Qwen2.5-1.5B-Instruct in 4-bit (NF4)

In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
from accelerate import init_empty_weights

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map={"":0},
    torch_dtype=torch.float16
)

with init_empty_weights():
    full_size = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID))
full_size.tie_weights()
print(f"Parameters: {full_size.num_parameters() / 1e9: .2f} B")
print(f"GPU memory used: {torch.cuda.memory_allocated(0) / 1e9: .2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Parameters:  1.54 B
GPU memory used:  1.15 GB


In [23]:
def n_tokens(prompt, completion):
    text = tokenizer.apply_chat_template(prompt + completion, tokenize=False)
    return len(tokenizer(text)["input_ids"])

lengths = train_df.apply(lambda r: n_tokens(r["prompt"], r["completion"]), axis=1)
print(lengths.describe())
print("Longest example:", lengths.max(), "tokens")

count    19396.000000
mean       167.935090
std         22.110875
min        137.000000
25%        154.000000
50%        163.000000
75%        175.000000
max        644.000000
dtype: float64
Longest example: 644 tokens


## 6. QLoRA fine-tuning
LoRA r=16, alpha=32, dropout 0.05 on attention and MLP projections; 1 epoch, effective batch 16, lr 2e-4 (cosine, 36 warmup steps), fp16 on a T4. Trainable LoRA weights are kept in float32.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_pandas(train_df[["prompt", "completion"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["prompt", "completion"]], preserve_index=False)

lora_config = LoraConfig(
    r=16,                      # rank: 16 signals per adapter
    lora_alpha=32,             # scaling = 2 x r
    lora_dropout=0.05,         # small dropout on adapters to reduce overfitting
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",      # attention layers
                    "gate_proj", "up_proj", "down_proj"],        # MLP layers
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="qwen-ade-lora",
    num_train_epochs=1,                 # one pass over ~19k examples
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 8 x 2 = 16
    learning_rate=2e-4,                 # recommended for LoRA
    lr_scheduler_type="cosine",         # learning rate slowly decreases
    warmup_steps=36,                  # small ramp-up at the start
    logging_steps=25,                   # print training loss every 25 steps
    eval_strategy="steps", eval_steps=200,   # validation loss every 200 steps
    save_strategy="steps", save_steps=200, save_total_limit=2,  # checkpoints, in case the session stops
    per_device_eval_batch_size=8,
    fp16=True,                          # 16-bit training (T4)
    bf16=False,
    gradient_checkpointing=True,        # saves memory by recomputing some values
    max_length=768,                     # longest example is 644 tokens
    completion_only_loss=True,          # loss only on the assistant answer
    report_to="none",
)

args._n_gpu = 1   # trainer use a single GPU

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=lora_config,            # the trainer attaches A and B, and W stays frozen
)

trainer.model.print_trainable_parameters()

# Trainable LoRA adapters in float32 (standard for QLoRA with fp16 training)
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
print({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})

Tokenizing train dataset:   0%|          | 0/19396 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/19396 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/19396 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/19396 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
{'torch.float32'}


In [ ]:
trainer.train()

*Training log from the original run* (1,213 steps, 3,211 s ≈ 54 min):

| Step | Training loss | Validation loss | Val token accuracy |
|---|---|---|---|
| 200 | 0.0337 | 0.0290 | 98.8% |
| 400 | 0.0307 | 0.0234 | 99.1% |
| 600 | 0.0278 | 0.0232 | 99.1% |
| 800 | 0.0127 | 0.0207 | 99.3% |
| 1000 | 0.0161 | 0.0198 | 99.3% |
| 1213 | 0.0211 | 0.0189 | 99.3% |

Validation loss fell at every checkpoint, so one epoch did not overfit. Token accuracy is inflated by the fixed JSON tokens; the real test is generation on unseen sentences (Section 11).

## 7. Save the adapter to Hugging Face

In [ ]:
ADAPTER_REPO = "AadarshKr/qwen2.5-1.5b-ade-lora"  # HF username / model name

trainer.save_model("qwen-ade-lora-final")          # local copy
trainer.model.push_to_hub(ADAPTER_REPO, private=True)   # permanent copy, private
tokenizer.push_to_hub(ADAPTER_REPO, private=True)
print("Saved and pushed:", ADAPTER_REPO)

## 8. Reload the adapter (only needed after a session restart)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("Logged in as:", whoami()["name"])

In [ ]:
from peft import PeftModel
ft_model = PeftModel.from_pretrained(model, "AadarshKr/qwen2.5-1.5b-ade-lora")
ft_model.config.use_cache = True
ft_model.eval()
print("Fine-tuned adapter loaded")

## 9. Evaluation functions
Generate (batched, greedy, left padding) -> parse the JSON (strict format check) -> score against the gold labels.

**Batched greedy generation**

In [26]:
def generate_answers(model, prompts, batch_size = 16, max_new_tokens = 512):
    tokenizer.padding_side = "left"
    model.eval()
    outputs = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(p, tokenize = False, add_generation_prompt=True) for p in batch]
        enc = tokenizer(texts, return_tensors ="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id = tokenizer.pad_token_id)
        new_tokens = gen[:, enc["input_ids"].shape[1]:]
        outputs += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        print(f"{min(i + batch_size, len(prompts))}/{len(prompts)}", end="\r")
    return outputs

**Parse the model output**

In [27]:
import re

def parse_answer(text):
    try:
        obj = json.loads(text.strip())
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match is None:
            return None
        try:
            obj = json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    if not isinstance(obj, dict):
        return None
    if not isinstance(obj.get("has_ade"), bool) or not isinstance(obj.get("pairs"), list):
        return None
    for pair in obj["pairs"]:
        if not(isinstance(pair,dict) and isinstance(pair.get("drug"), str) and isinstance(pair.get("effect"),str)):
            return None
    return obj

In [ ]:
print(parse_answer('{"has_ade": true, "pairs": [{"drug": "aspirin", "effect": "rash"}]}'))
print(parse_answer('Sure! Here it is: {"has_ade": false, "pairs": []}'))
print(parse_answer('I think there is an ADE.'))
print(parse_answer('{"has_ade": "yes", "pairs": []}'))  

{'has_ade': True, 'pairs': [{'drug': 'aspirin', 'effect': 'rash'}]}
{'has_ade': False, 'pairs': []}
None
None


**Scoring**
- JSON valid %
- ADE detection precision / recall / F1 (an invalid answer counts as "no ADE")
- Pair extraction precision / recall / F1, exact match after lowercasing and whitespace normalisation

In [31]:
from sklearn.metrics import precision_recall_fscore_support

def norm(s):
    return " ".join(s.lower().split())

def score(outputs, gold_df):  
    parsed = [parse_answer(o) for o in outputs]
    valid = [p is not None for p in parsed]

    y_true = gold_df["label"].astype(int).tolist()
    y_pred = [int(p["has_ade"]) if p is not None else 0 for p in parsed]
    p_ade, r_ade, f_ade, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

    tp = fp = fn = 0
    for p, gold_pairs in zip(parsed, gold_df["pairs"]):
        pred = {(norm(x["drug"]), norm(x["effect"])) for x in p["pairs"]} if p is not None else set()
        true = {(norm(x["drug"]), norm(x["effect"])) for x in gold_pairs}
        tp += len(pred & true); fp += len(pred - true); fn += len(true - pred)
    p_pair = tp / (tp + fp) if tp + fp else 0.0
    r_pair = tp / (tp + fn) if tp + fn else 0.0
    f_pair = 2 * p_pair * r_pair / (p_pair + r_pair) if p_pair + r_pair else 0.0

    metrics = {
        "json_valid_%": round(100 * sum(valid) / len(valid), 1),
        "ade_precision": round(p_ade, 3), "ade_recall": round(r_ade, 3), "ade_f1": round(f_ade, 3),
        "pair_precision": round(p_pair, 3), "pair_recall": round(r_pair, 3), "pair_f1": round(f_pair, 3),
    }
    return metrics, parsed

## 10. Smoke test (20 sentences)
*Output from the original run (preview only, 20 sentences):* fine-tuned ADE F1 0.889 / pair F1 0.909; base ADE F1 0.533 / pair F1 0.207; ~9 s per model.
The base model flagged non-ADE sentences (a disease, a device complication) as ADEs and wrapped its JSON in markdown fences, which `parse_answer` handles.

In [ ]:
import time
sample = test_df.head(20)

t0 = time.time()
ft_out = generate_answers(ft_model, sample["prompt"].tolist())
print("\nFine-tuned:", score(ft_out, sample)[0], f"({time.time() - t0:.0f}s)")

t0 = time.time()
with ft_model.disable_adapter():
    base_out = generate_answers(ft_model, sample["prompt"].tolist())
print("\nBase:", score(base_out, sample)[0], f"({time.time() - t0:.0f}s)")

for i in range(3):
    print("\nSENTENCE:", sample.iloc[i]["text"])
    print("TRUE:", sample.iloc[i]["completion"][0]["content"])
    print("BASE:", base_out[i][:300])
    print("FINE-TUNED:", ft_out[i])

## 11. Full evaluation: base vs fine-tuned (1,000 test sentences)
The base model is the same model with the adapter disabled, so the only difference is fine-tuning. Outputs and metrics are saved to the Hugging Face repo.
| Metric | Base | Fine-tuned |
|---|---|---|
| Valid JSON | 99.7% | **100%** |
| ADE precision | 0.40 | **0.93** |
| ADE recall | 0.82 | **0.88** |
| **ADE F1** | 0.54 | **0.90** |
| Pair precision | 0.18 | **0.70** |
| Pair recall | 0.49 | **0.69** |
| **Pair F1 (strict)** | 0.26 | **0.70** |

The base model flags most medical sentences as ADEs (precision 0.40). Fine-tuning removes most false alarms and also improves recall.

In [ ]:
import json
from huggingface_hub import HfApi

ft_out_all = generate_answers(ft_model, test_df["prompt"].tolist())
ft_metrics, ft_parsed = score(ft_out_all, test_df)
print("\nFine-tuned:", ft_metrics)

with ft_model.disable_adapter():
    base_out_all = generate_answers(ft_model, test_df["prompt"].tolist())
base_metrics, base_parsed = score(base_out_all, test_df)
print("\nBase:", base_metrics)

# save raw outputs + metrics permanently (for the results table and error analysis)
results = {"ft_metrics": ft_metrics, "base_metrics": base_metrics,
           "ft_outputs": ft_out_all, "base_outputs": base_out_all,
           "test_text": test_df["text"].tolist()}
with open("eval_results.json", "w") as f:
    json.dump(results, f)
HfApi().upload_file(path_or_fileobj="eval_results.json", path_in_repo="eval_results.json",
                    repo_id="AadarshKr/qwen2.5-1.5b-ade-lora")
print("Results saved to Hugging Face")

**Reload the saved results** (no regeneration needed)

In [32]:
import json
from huggingface_hub import hf_hub_download

path = hf_hub_download("AadarshKr/qwen2.5-1.5b-ade-lora", "eval_results.json")
with open(path) as f:
    results = json.load(f)

print("Fine-tuned:", results["ft_metrics"])
print("Base:", results["base_metrics"])
print(len(results["ft_outputs"]), "outputs loaded")

eval_results.json:   0%|          | 0.00/302k [00:00<?, ?B/s]

Fine-tuned: {'json_valid_%': 100.0, 'ade_precision': 0.932, 'ade_recall': 0.877, 'ade_f1': 0.904, 'pair_precision': 0.701, 'pair_recall': 0.694, 'pair_f1': 0.697}
Base: {'json_valid_%': 99.7, 'ade_precision': 0.4, 'ade_recall': 0.824, 'ade_f1': 0.538, 'pair_precision': 0.178, 'pair_recall': 0.489, 'pair_f1': 0.261}
1000 outputs loaded


## 12. Error analysis (fine-tuned model)

In [33]:
assert results["test_text"] == test_df["text"].tolist()
print("Order check passed")

ft_parsed = [parse_answer(o) for o in results["ft_outputs"]]

pred_ade = [p["has_ade"] if p is not None else False for p in ft_parsed]
pred_pairs = [p["pairs"] if p is not None else [] for p in ft_parsed]

ft_ea = pd.DataFrame({
    "text": test_df["text"].tolist(),
    "true_ade": test_df["label"].astype(bool).tolist(),
    "pred_ade": pred_ade,
    "true_pairs": test_df["pairs"].tolist(),
    "pred_pairs": pred_pairs
})

print(ft_ea.shape)
ft_ea.head()

Order check passed
(1000, 5)


,text,true_ade,pred_ade,true_pairs,pred_pairs
0,"However, with increased experience in applying...",False,False,[],[]
1,Chickenpox follows a severe and often fatal co...,False,False,[],[]
2,Catheter fracture and cardiac migration: a rar...,False,False,[],[]
3,"When retested 3 months later, at which time th...",False,False,[],[]
4,Discontinuation of the itraconazole caused res...,True,True,"[{'drug': 'itraconazole', 'effect': 'drug erup...","[{'drug': 'itraconazole', 'effect': 'drug erup..."


**Detection errors**

In [34]:
false_alarms = ft_ea[(ft_ea["pred_ade"]) & ~(ft_ea["true_ade"])]
misses = ft_ea[~(ft_ea["pred_ade"]) & (ft_ea["true_ade"])]

print(len(false_alarms))
print(len(misses))

13
25


In [35]:
false_alarms.head(5)

,text,true_ade,pred_ade,true_pairs,pred_pairs
31,"Hepatotoxicity seen with erlotinib, a small mo...",False,True,[],"[{'drug': 'erlotinib', 'effect': 'Hepatotoxici..."
157,To date little has been reported about the ris...,False,True,[],"[{'drug': 'etoposide', 'effect': 'leukaemia'},..."
169,CONCLUSIONS: Both the immunomodulatory effects...,False,True,[],"[{'drug': 'octreotide', 'effect': 'Graves' dis..."
218,"However, 19 days after starting amiodarone, se...",False,True,[],"[{'drug': 'amiodarone', 'effect': 'hypothyroid..."
314,The present patient was a 66-year-old Japanese...,False,True,[],"[{'drug': 'amiodarone', 'effect': 'cardiac arr..."


In [36]:
misses.head(5)

,text,true_ade,pred_ade,true_pairs,pred_pairs
10,Theoretical basal ganglia toxicologic mechanis...,True,False,"[{'drug': 'methanol', 'effect': 'methanol pois...",[]
58,Because the cerebellar toxicity may be worsene...,True,False,"[{'drug': 'HDARAC', 'effect': 'cerebellar toxi...",[]
95,These findings support previous studies that s...,True,False,"[{'drug': 'aspirin', 'effect': 'RS'}]",[]
108,All patients had taken phenytoin for variable ...,True,False,"[{'drug': 'phenytoin', 'effect': 'skin lesions'}]",[]
159,"CONCLUSION: To the best of our knowledge, this...",True,False,"[{'drug': 'colchicine', 'effect': 'colchicine ...",[]


In [37]:
for _, row in false_alarms.head(5).iterrows():
    print("TEXT:", row["text"])
    print("TRUE:", row["true_pairs"], "| PRED:", row["pred_pairs"], "\n")

TEXT: Hepatotoxicity seen with erlotinib, a small molecule tyrosine kinase inhibitor to EGFR, is usually transient with mild elevation of transaminases.
TRUE: [] | PRED: [{'drug': 'erlotinib', 'effect': 'Hepatotoxicity'}] 

TEXT: To date little has been reported about the risk of therapy-related leukaemia (t-AML) in children receiving oral etoposide therapy.
TRUE: [] | PRED: [{'drug': 'etoposide', 'effect': 'leukaemia'}, {'drug': 'etoposide', 'effect': 't-AML'}] 

TEXT: CONCLUSIONS: Both the immunomodulatory effects of octreotide on intrathyroidal lymphocytes and rapid reductions in TSH may contribute to the onset of Graves' disease.
TRUE: [] | PRED: [{'drug': 'octreotide', 'effect': "Graves' disease"}, {'drug': 'octreotide', 'effect': 'immunomodulatory effects'}] 

TEXT: However, 19 days after starting amiodarone, serum testing indicated a hypothyroid status that was not recognized clinically.
TRUE: [] | PRED: [{'drug': 'amiodarone', 'effect': 'hypothyroid status'}] 

TEXT: The presen

In [38]:
for _, row in misses.head(5).iterrows():
    print("TEXT:", row["text"])
    print("TRUE:", row["true_pairs"], "| PRED:", row["pred_pairs"], "\n")

TEXT: Theoretical basal ganglia toxicologic mechanisms of methanol poisoning are reviewed, and the role of brain imaging studies will regard to diagnosis, prognosis and impact on management is discussed.
TRUE: [{'drug': 'methanol', 'effect': 'methanol poisoning'}] | PRED: [] 

TEXT: Because the cerebellar toxicity may be worsened by continuation of therapy after initial onset of symptoms, prompt termination of HDARAC is recommended.
TRUE: [{'drug': 'HDARAC', 'effect': 'cerebellar toxicity'}] | PRED: [] 

TEXT: These findings support previous studies that showed that the use of aspirin during the antecedent illness may be a risk factor for the development of RS.
TRUE: [{'drug': 'aspirin', 'effect': 'RS'}] | PRED: [] 

TEXT: All patients had taken phenytoin for variable time periods (range 16-80 days; mean: 40) and were on the medication when the skin lesions first appeared.
TRUE: [{'drug': 'phenytoin', 'effect': 'skin lesions'}] | PRED: [] 

TEXT: CONCLUSION: To the best of our knowledg

**Findings**
- **False alarms (13):** a drug and a condition mentioned together without a causal claim: associations ("seen with", "risk of"), multi-factor causes, and indications (the drug is given *to treat* the condition).
- **Misses (25):** abbreviations (aspirin -> "RS", "HDARAC"), poisoning/intoxication counted as ADE (methanol, colchicine), hedged wording ("may be a risk factor").
- **Label inconsistency:** similar wording is labelled ADE in some sentences and not in others (e.g. phenytoin time-to-onset = ADE, amiodarone 19 days after start = not ADE), which limits achievable precision and recall.

**Extraction errors: strict vs lenient matching** (lenient = same drug and overlapping effect span)

In [39]:
def pairs_set(pairs):
    return {(norm(x["drug"]), norm(x["effect"])) for x in pairs}
    
def partial_match(p,t):
    p_drug, p_effect = (p)
    t_drug, t_effect = (t)
    return (t_drug == p_drug) and (t_effect in p_effect or p_effect in t_effect)

def classify_pairs(pred, true):
    exact = len(pred & true)
    pred_left = pred - true
    true_left = set(true - pred)
    boundary = wrong = 0
    for p in pred_left:
        match = None
        for t in true_left:
            if partial_match(p,t):
                match = t
                break
        if match is not None:
            boundary += 1
            true_left.remove(match)
        else:
            wrong += 1
    missed = len(true_left)    
    return exact, boundary, wrong, missed

In [40]:
totals = {"exact": 0, "boundary": 0, "wrong": 0, "missed": 0}
boundary_examples = []

for _, row in ft_ea.iterrows():
    pred = pairs_set(row["pred_pairs"])
    true = pairs_set(row["true_pairs"])
    e, b, w, m = classify_pairs(pred, true)
    totals["exact"] += e
    totals["boundary"] += b
    totals["wrong"] += w
    totals["missed"] += m
    if b > 0 and len(boundary_examples) < 5:  # keep a few examples to print below
        boundary_examples.append((row["text"], pred - true, true - pred))

def f1(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    return round(2 * p * r / (p + r), 3) if p + r else 0

E, B, W, M = totals["exact"], totals["boundary"], totals["wrong"], totals["missed"]
strict_f1  = f1(tp=E,     fp=B + W, fn=B + M)   # boundary counts as wrong AND missed
lenient_f1 = f1(tp=E + B, fp=W,     fn=M)       # boundary counts as correct
print("Strict pair F1:", strict_f1, "| Lenient pair F1:", lenient_f1)

for text, p, t in boundary_examples:
    print("\nTEXT:", text)
    print("PREDICTED:", p)
    print("TRUE:     ", t)

Strict pair F1: 0.697 | Lenient pair F1: 0.779

TEXT: Multiple pulmonary nodules: an unusual presentation of fludarabine pulmonary toxicity: case report and review of literature.
PREDICTED: {('fludarabine', 'pulmonary nodules')}
TRUE:      {('fludarabine', 'multiple pulmonary nodules')}

TEXT: Described here are 2 patients who developed thrombotic microangiopathy of the kidneys after receiving high cumulative doses of the new anticancer drug gemcitabine.
PREDICTED: {('gemcitabine', 'thrombotic microangiopathy of the kidneys')}
TRUE:      {('gemcitabine', 'thrombotic microangiopathy')}

TEXT: A 40-year-old man with a long standing history of rheumatoid arthritis was treated with MTX over a 6 month period and developed an overwhelming hepatic necrosis.
PREDICTED: {('mtx', 'hepatic necrosis')}
TRUE:      {('mtx', 'overwhelming hepatic necrosis')}

TEXT: A 73-year-old woman with non-Hodgkin's lymphoma had two episodes of severe, bilateral, sensori-neural hearing loss after vincristine ther

**Findings**
- Strict pair F1 **0.70**; lenient pair F1 **0.78**.
- The boundary cases are the same effect with or without modifiers ("hepatic necrosis" vs "overwhelming hepatic necrosis"). The dataset has no consistent rule for severity, location or quantity words, so strict F1 understates extraction quality.

## Conclusion
Fine-tuning Qwen2.5-1.5B with QLoRA (1.18% of parameters, ~54 min on a free T4) achieved the objective: ADE F1 0.54 -> 0.90 and pair F1 0.26 -> 0.70 on unseen sentences, with 100% valid JSON.
**Next steps:** Add training examples that separate “drug caused it” from “drug was given for it”; expand drug abbreviations before prediction; try LoRA ranks 8 and 32; compare against a few-shot prompt baseline.